In [1]:
import pandas as pd
import os
import re

base_path = "."

In [2]:
files = {
    "PM_diseases": "./data/raw/CTD_D052638_diseases_20260609050501.csv",
    "PM_genes": "./data/raw/CTD_D052638_genes_20260609050446.csv",
    "PM_ixns": "./data/raw/CTD_D052638_ixns_20260609050415.csv",
    "PM_phenotype": "./data/raw/CTD_D052638_phenotype_20260609050515.csv",
}

In [3]:

def read_ctd_csv(filename):
    return pd.read_csv(os.path.join(base_path, filename), comment="#")

pm_diseases = read_ctd_csv(files["PM_diseases"])
pm_genes = read_ctd_csv(files["PM_genes"])
pm_ixns = read_ctd_csv(files["PM_ixns"])
pm_phenotype = read_ctd_csv(files["PM_phenotype"])

In [4]:
# Audit global
for name, df in {
    "PM diseases": pm_diseases,
    "PM genes": pm_genes,
    "PM gene interactions": pm_ixns,
    "PM phenotype": pm_phenotype,
}.items():
    print("\n", name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())


 PM diseases
Shape: (17516, 9)
Columns: ['Chemical Name', 'Chemical ID', 'CAS RN', 'Disease Name', 'Disease ID', 'Direct Evidence', 'Inference Network', 'Inference Score', 'Reference Count']

 PM genes
Shape: (20155, 4)
Columns: ['Gene Symbol', 'Gene ID', 'Interaction Count', 'Organism Count']

 PM gene interactions
Shape: (66552, 9)
Columns: ['Chemical Name', 'Chemical ID', 'CAS RN', 'Gene Symbol', 'Gene ID', 'Interaction', 'Interaction Actions', 'References', 'Organisms']

 PM phenotype
Shape: (4830, 12)
Columns: ['Chemical', 'Chemical ID', 'CAS RN', 'Phenotype', 'Phenotype ID', 'Co-Mentioned Terms', 'Interaction', 'Organisms', 'Anatomy', 'Inference Network', 'Reference Count', 'References']


In [7]:
# Répartition des chemicals
print(pm_diseases["Chemical Name"].value_counts().head(20))
print("---" * 10)
print(pm_ixns["Chemical Name"].value_counts().head(20))
print("---" * 10)
print(pm_phenotype["Chemical"].value_counts().head(20))

Chemical Name
Particulate Matter         4615
Tobacco Smoke Pollution    4481
Smoke                      3288
Soot                       2701
Dust                       1295
Coal Ash                    747
Cosmic Dust                 385
Smog                          4
Name: count, dtype: int64
------------------------------
Chemical Name
Particulate Matter         29697
Tobacco Smoke Pollution    19111
Smoke                      10157
Soot                        6317
Dust                         977
Coal Ash                     252
Cosmic Dust                   41
Name: count, dtype: int64
------------------------------
Chemical
Particulate Matter         3215
Tobacco Smoke Pollution     986
Soot                        241
Smoke                       194
Dust                        163
Coal Ash                     29
Cosmic Dust                   2
Name: count, dtype: int64


In [9]:
# Top gènes
print(pm_genes.sort_values("Interaction Count", ascending=False).head(20))

   Gene Symbol  Gene ID  Interaction Count  Organism Count
0          IL6     3569                677               5
1          TNF     7124                562               6
2         IL1B     3553                489               6
3        CXCL8     3576                398               5
4       NFE2L2     4780                283               3
5        HMOX1     3162                256               4
6       CYP1A1     1543                235               5
7         RELA     5970                202               4
8         APOE      348                201               2
9         CCL2     6347                179               4
10         RAF    31221                172               1
11       PTGS2     5743                164               4
12       CXCL1     2919                154               3
13       MAPK1     5594                153               3
14       MAPK3     5595                147               3
15         APP      351                146              

In [11]:
# Interactions humaines uniquement
pm_ixns_human = pm_ixns[
    pm_ixns["Organisms"].astype(str).str.contains("Homo sapiens", na=False)
].copy()

print("Lignes humaines :", len(pm_ixns_human))
print("--" * 10)
print("Gènes humains uniques :", pm_ixns_human["Gene Symbol"].nunique())
print("--" * 10)
print(pm_ixns_human["Gene Symbol"].value_counts().head(20))

Lignes humaines : 42243
--------------------
Gènes humains uniques : 16805
--------------------
Gene Symbol
CXCL8     217
IL6       175
TNF       121
IL1B      118
RELA       90
HMOX1      79
CYP1A1     72
MMP1       66
NFE2L2     66
PTGS2      62
MAPK1      60
NQO1       60
MUC5AC     59
MAPK3      56
CDH1       49
CSF2       49
CDKN1A     48
ICAM1      48
CASP3      46
AHR        44
Name: count, dtype: int64


In [12]:
# Extraction des PMID depuis la colonne References
def extract_pmids(series):
    pmids = set()
    for value in series.dropna().astype(str):
        for ref in value.split("|"):
            match = re.search(r"\d+", ref)
            if match:
                pmids.add(match.group(0))
    return pmids

ixns_pmids = extract_pmids(pm_ixns["References"])
phenotype_pmids = extract_pmids(pm_phenotype["References"])

print("PMID uniques interactions :", len(ixns_pmids))
print("PMID uniques phenotype :", len(phenotype_pmids))

PMID uniques interactions : 1290
PMID uniques phenotype : 909


In [13]:
# Actions biologiques
actions = []

for value in pm_ixns["Interaction Actions"].dropna().astype(str):
    for action in value.split("|"):
        action = action.strip()
        if action:
            actions.append(action)

top_actions = pd.Series(actions).value_counts().head(30)
print(top_actions)

increases^expression               29749
decreases^expression               28629
increases^abundance                24251
affects^cotreatment                 8224
affects^expression                  4995
decreases^reaction                  3683
increases^reaction                  1572
increases^response to substance      825
affects^reaction                     796
increases^secretion                  734
affects^response to substance        654
increases^phosphorylation            506
increases^methylation                463
decreases^methylation                390
increases^activity                   307
affects^binding                      165
affects^methylation                  150
increases^cleavage                   142
affects^localization                 105
decreases^activity                   101
decreases^secretion                   68
decreases^phosphorylation             62
increases^metabolic processing        54
decreases^abundance                   27
decreases^nitros

In [14]:
# Top phénotypes
print(pm_phenotype["Phenotype"].value_counts().head(30))

Phenotype
reactive oxygen species metabolic process                                       161
leukocyte migration involved in inflammatory response                           124
cell population proliferation                                                   102
cellular metabolic process                                                      102
positive regulation of reactive oxygen species biosynthetic process             100
apoptotic process                                                                95
lipid oxidation                                                                  95
DNA methylation on cytosine within a CG sequence                                 91
reactive oxygen species biosynthetic process                                     81
glutathione metabolic process                                                    80
inflammatory response                                                            80
detection of oxidative stress                                     

In [15]:
# Phénotypes humains uniquement
pm_phenotype_human = pm_phenotype[
    pm_phenotype["Organisms"].astype(str).str.contains("Homo sapiens", na=False)
].copy()

print("Lignes phenotype humaines :", len(pm_phenotype_human))
print("Phénotypes humains uniques :", pm_phenotype_human["Phenotype"].nunique())
print(pm_phenotype_human["Phenotype"].value_counts().head(30))

Lignes phenotype humaines : 2187
Phénotypes humains uniques : 432
Phenotype
reactive oxygen species metabolic process                              123
cellular metabolic process                                              78
DNA methylation on cytosine within a CG sequence                        73
reactive oxygen species biosynthetic process                            69
positive regulation of reactive oxygen species biosynthetic process     64
apoptotic process                                                       59
cell population proliferation                                           45
lipid oxidation                                                         40
positive regulation of myeloid dendritic cell activation                36
glutathione metabolic process                                           34
negative regulation of cell population proliferation                    33
positive regulation of cell death                                       28
positive regulation of a